# CNN-Transformer

The convolutional front end of the CNN-LSTM followed by two Transformer encoder blocks with sinusoidal positional encoding, on windows of 50 frames with 52 antenna-averaged amplitudes each. The number of epochs is chosen on inner folds that hold out whole training clusters. Every split runs in its own process (`cnn_transformer_core.py`), and results are written to `results/`.

## Step 1: Settings

In [ ]:
import os
import json
import sys
import importlib

import matplotlib.pyplot as plt
import numpy as np
import keras
from sklearn.metrics import (accuracy_score, average_precision_score, classification_report, confusion_matrix,
                             precision_recall_curve, roc_auc_score, roc_curve)

import cnn_transformer_core as core
importlib.reload(core)

print("Python", sys.version.split()[0], "| Keras", keras.__version__, "|", sys.executable)

DATA_DIR = os.path.abspath("../output_raw_full")
RESULTS_DIR = "results"
OTHER_RESULTS = {"SVM": "../SVM/results", "RF": "../RF/results",
                 "CNN-LSTM": "../CNN_LSTM/results"}

CONFIG = dict(core.DEFAULT_CONFIG)
CONFIG.update({
    "input": "subcarrier",
    "n_pca": 64,
    "conv_filters": [64, 128], "kernel_size": 3, "pool_size": 2, "padding": "valid", "conv_dropout": 0.1,
    "d_model": None, "pos_encoding": "sinusoidal",
    "n_blocks": 2, "num_heads": 4, "ff_dim": 128, "attn_dropout": 0.1,
    "dense_units": 32, "l2": 1e-3, "dropout": 0.4,
    "noise_std": 0.0,
    "learning_rate": 1e-3, "batch_size": 32, "max_epochs": 50, "inner_patience": 10,
    "window": 50, "stride": 10, "seed": 42,
    "data_dir": DATA_DIR,
})
for k, v in CONFIG.items():
    print(f"  {k:15s} {v}")

CLASS_NAMES = ["NLOS (0)", "LOS (1)"]

COLOR_LOS = "#2a78d6"
COLOR_NLOS = "#eb6834"
COLOR_VAL = "#1baf7a"
INK = "#0b0b0b"
INK_2 = "#52514e"
GRID = "#e4e3df"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": INK_2, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11, "lines.linewidth": 2,
})
os.makedirs(RESULTS_DIR, exist_ok=True)

## Step 2: Load the per-cluster features

In [ ]:
LOS_CLUSTERS = ["LOS_cluster1", "LOS_cluster2", "LOS_cluster3", "LOS_cluster4"]
NLOS_CLUSTERS = ["NLOS_cluster1", "NLOS_cluster2", "NLOS_cluster3", "NLOS_cluster4", "NLOS_cluster5"]
ALL_CLUSTERS = LOS_CLUSTERS + NLOS_CLUSTERS

frames, labels = core.load_cluster_frames(DATA_DIR, ALL_CLUSTERS, CONFIG["input"])
W, S = CONFIG["window"], CONFIG["stride"]
for c in ALL_CLUSTERS:
    n_win = (len(frames[c]) - W) // S + 1
    print(f"{c:14s} label={labels[c]}  frames={frames[c].shape}  windows={n_win}")

## Step 3: Splits

`lopo` holds out one LOS and one NLOS cluster (20 splits). `l2po` is the L2PO protocol and holds out two LOS and two NLOS clusters (60 splits).

In [ ]:
import itertools

PROTOCOLS = ["lopo", "l2po"]
short = lambda c: c.replace("_cluster", "")

def make_splits(protocol):
    if protocol == "lopo":
        tests = [[l, n] for l in LOS_CLUSTERS for n in NLOS_CLUSTERS]
    elif protocol == "l2po":
        tests = [list(ls) + list(ns) for ls in itertools.combinations(LOS_CLUSTERS, 2)
                 for ns in itertools.combinations(NLOS_CLUSTERS, 2)]
    else:
        raise ValueError(protocol)
    return [{"name": "test=" + "+".join(short(c) for c in test),
             "train": [c for c in ALL_CLUSTERS if c not in test], "test": test} for test in tests]

SPLITS = {p: make_splits(p) for p in PROTOCOLS}
for p, splits in SPLITS.items():
    print(f"{p}: {len(splits)} splits")
    for i, sp in enumerate(splits, 1):
        print(f"  {i:2d}. train={'+'.join(short(c) for c in sp['train']):38s} {sp['name']}")

## Step 4: Worker for one split

Runs `cnn_transformer_core.py` in a separate process with the settings of Step 1, saved to `results/part_b_config.json`.

In [ ]:
import subprocess

WORKER = os.path.abspath("cnn_transformer_core.py")
PART_B_CONFIG = os.path.join(RESULTS_DIR, "part_b_config.json")
with open(PART_B_CONFIG, "w") as f:
    json.dump(CONFIG, f, indent=2)
print("worker :", WORKER)
print("python :", sys.executable)
print("config :", PART_B_CONFIG)

def evaluate_split(split):
    arg = json.dumps({k: split[k] for k in ("name", "train", "test")})
    proc = subprocess.run([sys.executable, WORKER, "--config", PART_B_CONFIG, "--split", arg],
                          capture_output=True, text=True)
    lines = [l for l in proc.stdout.splitlines() if l.startswith("RESULT ")]
    if proc.returncode != 0 or not lines:
        raise RuntimeError(f"split {split['name']} failed (exit code {proc.returncode}):\n{proc.stderr[-3000:]}")
    return json.loads(lines[-1][len("RESULT "):])

## Step 5: Run all splits

Each split is appended to `results/splits_<protocol>.jsonl`, and finished splits are skipped when the cell runs again. Delete these files after changing a setting.

In [ ]:
import time

def load_done(path):
    if not os.path.exists(path):
        return {}
    with open(path) as f:
        return {r["name"]: r for r in (json.loads(line) for line in f if line.strip())}

ALL_RESULTS = {}
for protocol, splits in SPLITS.items():
    path = os.path.join(RESULTS_DIR, f"splits_{protocol}.jsonl")
    done = load_done(path)
    print(f"=== {protocol}: {len(splits)} splits ({len(done)} already done)")
    t_start = time.time()
    for i, sp in enumerate(splits, 1):
        if sp["name"] in done:
            continue
        r = evaluate_split(sp)
        with open(path, "a") as f:
            f.write(json.dumps(r) + "\n")
        done[r["name"]] = r
        print(f"  [{i:2d}/{len(splits)}] {r['name']:32s} acc={r['accuracy']:.4f}  F1={r['macro_f1']:.4f}  "
              f"AUC={r['roc_auc']:.4f}  inner={r['inner_cv_accuracy']:.4f}  epochs={r['selected_epochs']}  "
              f"({time.time() - t_start:.0f}s total)")
    ALL_RESULTS[protocol] = [done[sp["name"]] for sp in splits]

## Step 6: Summary

Includes the inner held-out accuracy, which is estimated from the training clusters only.

In [ ]:
SUMMARY = {}
for protocol, res in ALL_RESULTS.items():
    acc_s = np.array([r["accuracy"] for r in res])
    f1_s = np.array([r["macro_f1"] for r in res])
    auc_s = np.array([r["roc_auc"] for r in res])
    inner_s = np.array([r["inner_cv_accuracy"] for r in res])
    ep_s = np.array([r["selected_epochs"] for r in res])
    worst = res[int(acc_s.argmin())]
    print(f"=== {protocol} ({len(res)} splits)")
    print(f"  accuracy : {acc_s.mean():.4f} ± {acc_s.std():.4f}   min {acc_s.min():.4f}   max {acc_s.max():.4f}")
    print(f"  macro-F1 : {f1_s.mean():.4f} ± {f1_s.std():.4f}")
    print(f"  ROC-AUC  : {auc_s.mean():.4f} ± {auc_s.std():.4f}")
    print(f"  inner held-out accuracy: {inner_s.mean():.4f}  -> gap to test {inner_s.mean() - acc_s.mean():+.4f}")
    print(f"  selected epochs: mean {ep_s.mean():.1f} (range {ep_s.min()}–{ep_s.max()})")
    print(f"  worst split: {worst['name']}  (acc {worst['accuracy']:.4f})")

    held = {c: [r["per_cluster_accuracy"][c] for r in res if c in r["per_cluster_accuracy"]] for c in ALL_CLUSTERS}
    held = {c: v for c, v in held.items() if v}
    print("  accuracy of each cluster when held out (mean / worst / number of splits):")
    for c in held:
        print(f"    {c:14s} {np.mean(held[c]):.4f} / {np.min(held[c]):.4f} / {len(held[c])}")

    entry = {"n_splits": len(res),
             "accuracy_mean": float(acc_s.mean()), "accuracy_std": float(acc_s.std()),
             "accuracy_min": float(acc_s.min()), "accuracy_max": float(acc_s.max()),
             "macro_f1_mean": float(f1_s.mean()), "macro_f1_std": float(f1_s.std()),
             "roc_auc_mean": float(auc_s.mean()), "roc_auc_std": float(auc_s.std()),
             "inner_cv_accuracy_mean": float(inner_s.mean()), "selected_epochs_mean": float(ep_s.mean()),
             "worst_split": worst["name"],
             "held_out_cluster_accuracy": {c: {"mean": float(np.mean(v)), "min": float(np.min(v)), "n": len(v)}
                                           for c, v in held.items()}}
    SUMMARY[protocol] = entry
    print()

## Step 7: Figures and saved summary

In [ ]:
for protocol, res in ALL_RESULTS.items():
    acc_s = np.array([r["accuracy"] for r in res])
    order = np.argsort(acc_s)
    fig, ax = plt.subplots(figsize=(8, 3.8))
    ax.plot(np.arange(len(acc_s)), acc_s[order], "o", color=COLOR_LOS, markersize=6)
    ax.axhline(acc_s.mean(), color=INK_2, linestyle="--", linewidth=1)
    ax.text(0, acc_s.mean(), f" mean {acc_s.mean():.3f}", va="bottom", color=INK_2)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.set_xlabel("Split (sorted by accuracy)")
    ax.set_ylabel("Test accuracy")
    ax.set_title(f"CNN-Transformer: accuracy of all {len(acc_s)} {protocol} splits")
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTS_DIR, f"splits_{protocol}_accuracy.png"), dpi=150)
    plt.show()

    held = SUMMARY[protocol]["held_out_cluster_accuracy"]
    shown = [c for c in ALL_CLUSTERS if c in held]
    means = np.array([held[c]["mean"] for c in shown])
    mins = np.array([held[c]["min"] for c in shown])
    maxs = np.array([max(r["per_cluster_accuracy"][c] for r in res if c in r["per_cluster_accuracy"]) for c in shown])
    colors = [COLOR_LOS if c.startswith("LOS") else COLOR_NLOS for c in shown]
    fig, ax = plt.subplots(figsize=(8, 4.6))
    y = np.arange(len(shown))
    ax.set_axisbelow(True)
    ax.grid(axis="y", visible=False)
    ax.barh(y, means, color=colors, height=0.6, edgecolor="#fcfcfb", linewidth=2)
    ax.errorbar(means, y, xerr=[means - mins, maxs - means], fmt="none", ecolor=INK_2, capsize=3, linewidth=1)
    for yi, m, hi in zip(y, means, maxs):
        ax.text(hi + 0.015, yi, f"{m:.3f}", va="center", color=INK, fontsize=9)
    ax.set_yticks(y, [short(c) for c in shown])
    ax.invert_yaxis()
    ax.set_xlim(0, 1.12)
    ax.set_xlabel("Accuracy when the cluster is held out (mean; bar = worst to best split)")
    ax.set_title(f"CNN-Transformer ({protocol}): accuracy per held-out cluster")
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=COLOR_LOS), plt.Rectangle((0, 0), 1, 1, color=COLOR_NLOS)],
              labels=["LOS cluster", "NLOS cluster"], frameon=False, ncol=2,
              loc="upper center", bbox_to_anchor=(0.5, -0.16))
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTS_DIR, f"splits_{protocol}_per_cluster.png"), dpi=150)
    plt.show()

with open(os.path.join(RESULTS_DIR, "splits_summary.json"), "w") as f:
    json.dump({"config": CONFIG, "protocols": SUMMARY}, f, indent=2)
print("saved:", os.path.join(RESULTS_DIR, "splits_summary.json"))

## Step 8: Comparison with SVM, RF and CNN-LSTM on the same splits

In [ ]:
from scipy.stats import wilcoxon

ME = "CNN-Transformer"
for protocol, res in ALL_RESULTS.items():
    mine = {r["name"]: r for r in res}
    table = {ME: mine}
    for name, d in OTHER_RESULTS.items():
        path = os.path.join(d, f"splits_{protocol}.jsonl")
        if os.path.exists(path):
            table[name] = load_done(path)
    common = [n for n in mine if all(n in t for t in table.values())]
    print(f"=== {protocol}: {len(common)} splits available for {list(table)}")
    for name, t in table.items():
        a = np.array([t[n]["accuracy"] for n in common])
        print(f"  {name:16s} accuracy {a.mean():.4f} ± {a.std():.4f}   min {a.min():.4f}")
    for name in table:
        if name == ME:
            continue
        d = np.array([mine[n]["accuracy"] - table[name][n]["accuracy"] for n in common])
        nz = d[d != 0]
        p = wilcoxon(nz).pvalue if len(nz) >= 6 else float("nan")
        print(f"  {ME} - {name}: {d.mean():+.4f} on average; better on {(d > 0).sum()}, worse on {(d < 0).sum()}, "
              f"equal on {(d == 0).sum()} splits; Wilcoxon p = {p:.4g}")
    print(f"  {'held-out cluster':16s}" + "".join(f"{n[:14]:>16s}" for n in table))
    for c in ALL_CLUSTERS:
        vals = []
        for name, t in table.items():
            v = [t[n]["per_cluster_accuracy"][c] for n in common if c in t[n]["per_cluster_accuracy"]]
            vals.append(np.mean(v) if v else float("nan"))
        print(f"  {c:16s}" + "".join(f"{v:16.4f}" for v in vals))
    print()